In [1]:
import os
import io

import numpy as np

from typing import Tuple

import time
import cv2

from PIL import Image, ImageOps

import torch
import torchvision
import torch.onnx

from onnx_tf.backend import prepare
import onnx
from onnxsim import simplify

# import tvm
# import tvm.relay
# import tvm.contrib.graph_runtime as graph_runtime

from mobilenet_v2_tsm import MobileNetV2

# import warnings
# warnings.filterwarnings('ignore')

/Users/izakharkin/Desktop/skoltech/vrarhaptics/deepjest/phynder/convert/onnx-tensorflow/onnx_tf/common/__init__.py:96: UserWarning: onnx_tf.common.get_outputs_names is deprecated. It will be removed in future release. Use TensorflowGraph.get_outputs_names instead.
  warnings.warn(message)


In [2]:
SOFTMAX_THRES = 0
HISTORY_LOGIT = True
REFINE_OUTPUT = True

# def torch2tvm_module(torch_module: torch.nn.Module, torch_inputs: Tuple[torch.Tensor, ...], target):
#     torch_module.eval()
#     input_names = []
#     input_shapes = {}
#     with torch.no_grad():
#         for index, torch_input in enumerate(torch_inputs):
#             name = "i" + str(index)
#             input_names.append(name)
#             input_shapes[name] = torch_input.shape
#         buffer = io.BytesIO()
#         torch.onnx.export(torch_module, torch_inputs, buffer, input_names=input_names, output_names=["o" + str(i) for i in range(len(torch_inputs))])
#         outs = torch_module(*torch_inputs)
#         buffer.seek(0, 0)
#         onnx_model = onnx.load_model(buffer)
#         relay_module, params = tvm.relay.frontend.from_onnx(onnx_model, shape=input_shapes)
#     with tvm.relay.build_config(opt_level=3):
#         graph, tvm_module, params = tvm.relay.build(relay_module, target, params=params)
#     return graph, tvm_module, params


# def torch2executor(torch_module: torch.nn.Module, torch_inputs: Tuple[torch.Tensor, ...], target):
#     prefix = f"mobilenet_tsm_tvm_{target}"
#     lib_fname = f'{prefix}.tar'
#     graph_fname = f'{prefix}.json'
#     params_fname = f'{prefix}.params'
#     if os.path.exists(lib_fname) and os.path.exists(graph_fname) and os.path.exists(params_fname):
#         with open(graph_fname, 'rt') as f:
#             graph = f.read()
#         tvm_module = tvm.module.load(lib_fname)
#         params = tvm.relay.load_param_dict(bytearray(open(params_fname, 'rb').read()))
#     else:
#         graph, tvm_module, params = torch2tvm_module(torch_module, torch_inputs, target)
#         tvm_module.export_library(lib_fname)
#         with open(graph_fname, 'wt') as f:
#             f.write(graph)
#         with open(params_fname, 'wb') as f:
#             f.write(tvm.relay.save_param_dict(params))

#     ctx = tvm.gpu() if target.startswith('cuda') else tvm.cpu()
#     graph_module = graph_runtime.create(graph, tvm_module, ctx)
#     for pname, pvalue in params.items():
#         graph_module.set_input(pname, pvalue)

#     def executor(inputs: Tuple[tvm.nd.NDArray]):
#         for index, value in enumerate(inputs):
#             graph_module.set_input(index, value)
#         graph_module.run()
#         return tuple(graph_module.get_output(index) for index in range(len(inputs)))

#     return executor, ctx


# def get_executor(use_gpu=True):
#     torch_module = MobileNetV2(n_class=27)
#     if not os.path.exists("mobilenetv2_jester_online.pth.tar"):  # checkpoint not downloaded
#         print('Downloading PyTorch checkpoint...')
#         import urllib.request
#         url = 'https://file.lzhu.me/projects/tsm/models/mobilenetv2_jester_online.pth.tar'
#         urllib.request.urlretrieve(url, './mobilenetv2_jester_online.pth.tar')
#     torch_module.load_state_dict(torch.load("mobilenetv2_jester_online.pth.tar"))
#     torch_inputs = (torch.rand(1, 3, 224, 224),
#                     torch.zeros([1, 3, 56, 56]),
#                     torch.zeros([1, 4, 28, 28]),
#                     torch.zeros([1, 4, 28, 28]),
#                     torch.zeros([1, 8, 14, 14]),
#                     torch.zeros([1, 8, 14, 14]),
#                     torch.zeros([1, 8, 14, 14]),
#                     torch.zeros([1, 12, 14, 14]),
#                     torch.zeros([1, 12, 14, 14]),
#                     torch.zeros([1, 20, 7, 7]),
#                     torch.zeros([1, 20, 7, 7]))
#     if use_gpu:
#         target = 'cuda'
#     else:
#         target = 'llvm -mcpu=cortex-a72 -target=armv7l-linux-gnueabihf'
#     return torch2executor(torch_module, torch_inputs, target)


def transform(frame: np.ndarray):
    # 480, 640, 3, 0 ~ 255
    frame = cv2.resize(frame, (224, 224))  # (224, 224, 3) 0 ~ 255
    frame = frame / 255.0  # (224, 224, 3) 0 ~ 1.0
    frame = np.transpose(frame, axes=[2, 0, 1])  # (3, 224, 224) 0 ~ 1.0
    frame = np.expand_dims(frame, axis=0)  # (1, 3, 480, 640) 0 ~ 1.0
    return frame


class GroupScale(object):
    """ Rescales the input PIL.Image to the given 'size'.
    'size' will be the size of the smaller edge.
    For example, if height > width, then image will be
    rescaled to (size * height / width, size)
    size: size of the smaller edge
    interpolation: Default: PIL.Image.BILINEAR
    """

    def __init__(self, size, interpolation=Image.BILINEAR):
        self.worker = torchvision.transforms.Scale(size, interpolation)

    def __call__(self, img_group):
        return [self.worker(img) for img in img_group]


class GroupCenterCrop(object):
    def __init__(self, size):
        self.worker = torchvision.transforms.CenterCrop(size)

    def __call__(self, img_group):
        return [self.worker(img) for img in img_group]


class Stack(object):

    def __init__(self, roll=False):
        self.roll = roll

    def __call__(self, img_group):
        if img_group[0].mode == 'L':
            return np.concatenate([np.expand_dims(x, 2) for x in img_group], axis=2)
        elif img_group[0].mode == 'RGB':
            if self.roll:
                return np.concatenate([np.array(x)[:, :, ::-1] for x in img_group], axis=2)
            else:
                return np.concatenate(img_group, axis=2)


class ToTorchFormatTensor(object):
    """ Converts a PIL.Image (RGB) or numpy.ndarray (H x W x C) in the range [0, 255]
    to a torch.FloatTensor of shape (C x H x W) in the range [0.0, 1.0] """

    def __init__(self, div=True):
        self.div = div

    def __call__(self, pic):
        if isinstance(pic, np.ndarray):
            # handle numpy array
            img = torch.from_numpy(pic).permute(2, 0, 1).contiguous()
        else:
            # handle PIL Image
            img = torch.ByteTensor(torch.ByteStorage.from_buffer(pic.tobytes()))
            img = img.view(pic.size[1], pic.size[0], len(pic.mode))
            # put it from HWC to CHW format
            # yikes, this transpose takes 80% of the loading time/CPU
            img = img.transpose(0, 1).transpose(0, 2).contiguous()
        return img.float().div(255) if self.div else img.float()


class GroupNormalize(object):
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        rep_mean = self.mean * (tensor.size()[0] // len(self.mean))
        rep_std = self.std * (tensor.size()[0] // len(self.std))

        # TODO: make efficient
        for t, m, s in zip(tensor, rep_mean, rep_std):
            t.sub_(m).div_(s)

        return tensor


def get_transform():
    cropping = torchvision.transforms.Compose([
        GroupScale(256),
        GroupCenterCrop(224),
    ])
    transform = torchvision.transforms.Compose([
        cropping,
        Stack(roll=False),
        ToTorchFormatTensor(div=True),
        GroupNormalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return transform

catigories = [
    "Doing other things",  # 0
    "Drumming Fingers",  # 1
    "No gesture",  # 2
    "Pulling Hand In",  # 3
    "Pulling Two Fingers In",  # 4
    "Pushing Hand Away",  # 5
    "Pushing Two Fingers Away",  # 6
    "Rolling Hand Backward",  # 7
    "Rolling Hand Forward",  # 8
    "Shaking Hand",  # 9
    "Sliding Two Fingers Down",  # 10
    "Sliding Two Fingers Left",  # 11
    "Sliding Two Fingers Right",  # 12
    "Sliding Two Fingers Up",  # 13
    "Stop Sign",  # 14
    "Swiping Down",  # 15
    "Swiping Left",  # 16
    "Swiping Right",  # 17
    "Swiping Up",  # 18
    "Thumb Down",  # 19
    "Thumb Up",  # 20
    "Turning Hand Clockwise",  # 21
    "Turning Hand Counterclockwise",  # 22
    "Zooming In With Full Hand",  # 23
    "Zooming In With Two Fingers",  # 24
    "Zooming Out With Full Hand",  # 25
    "Zooming Out With Two Fingers"  # 26
]


n_still_frame = 0

def process_output(idx_, history):
    # idx_: the output of current frame
    # history: a list containing the history of predictions
    if not REFINE_OUTPUT:
        return idx_, history

    max_hist_len = 20  # max history buffer

    # mask out illegal action
    if idx_ in [7, 8, 21, 22, 3]:
        idx_ = history[-1]

    # use only single no action class
    if idx_ == 0:
        idx_ = 2
    
    # history smoothing
    if idx_ != history[-1]:
        if not (history[-1] == history[-2]): #  and history[-2] == history[-3]):
            idx_ = history[-1]
    

    history.append(idx_)
    history = history[-max_hist_len:]

    return history[-1], history

In [3]:
def torch2onnx(
    torch_module: torch.nn.Module, 
    torch_inputs: Tuple[torch.Tensor, ...], 
    onnx_path
):
    torch_module.eval()
    input_names = []
    input_shapes = {}
    with torch.no_grad():
        for index, torch_input in enumerate(torch_inputs):
            name = "i" + str(index)
            input_names.append(name)
            input_shapes[name] = torch_input.shape
        buffer = io.BytesIO()
        with open(onnx_path, 'wb') as model_file:
            torch.onnx.export(
                torch_module, 
                torch_inputs, 
                model_file, 
                input_names=input_names, 
                output_names=["o" + str(i) for i in range(len(torch_inputs))],
                opset_version=10
            )

* Load the model:

In [4]:
os.makedirs('./models', exist_ok=True)
TORCH_MODEL_PATH= './models/mobilenetv2_jester_online.pth.tar'
torch_module = MobileNetV2(n_class=27)
if not os.path.exists(TORCH_MODEL_PATH):  # checkpoint not downloaded
    print('Downloading PyTorch checkpoint...')
    import urllib.request
    url = 'https://file.lzhu.me/projects/tsm/models/mobilenetv2_jester_online.pth.tar'
    urllib.request.urlretrieve(url, TORCH_MODEL_PATH)
torch_module.load_state_dict(torch.load(TORCH_MODEL_PATH))
torch_module.eval()

MobileNetV2(
  (features): ModuleList(
    (0): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
        (3): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
       

* Convert to ONNX:

In [5]:
ONNX_MODEL_PATH = './models/jestnet.onnx'

torch_inputs = (torch.rand(1, 3, 224, 224, dtype=torch.float32),
                torch.zeros([1, 3, 56, 56], dtype=torch.float32),
                torch.zeros([1, 4, 28, 28], dtype=torch.float32),
                torch.zeros([1, 4, 28, 28], dtype=torch.float32),
                torch.zeros([1, 8, 14, 14], dtype=torch.float32),
                torch.zeros([1, 8, 14, 14], dtype=torch.float32),
                torch.zeros([1, 8, 14, 14], dtype=torch.float32),
                torch.zeros([1, 12, 14, 14], dtype=torch.float32),
                torch.zeros([1, 12, 14, 14], dtype=torch.float32),
                torch.zeros([1, 20, 7, 7], dtype=torch.float32),
                torch.zeros([1, 20, 7, 7], dtype=torch.float32))

for param in torch_module.parameters():
    param = param.float()

for module in torch_module.children():
    for module_1 in module.children():
        for module_2 in module_1.children():
            for module_3 in module_2.children():
                if hasattr(module_3, 'num_batches_tracked'):
                    module_3.num_batches_tracked = module_3.num_batches_tracked.float()
#                 if str(module_3).split('(')[0] == 'BatchNorm2d':
#                     print(module_3)

torch2onnx(
    torch_module=torch_module, 
    torch_inputs=torch_inputs, 
    onnx_path=ONNX_MODEL_PATH
)

/Users/izakharkin/Desktop/skoltech/vrarhaptics/deepjest/phynder/convert/mobilenet_v2_tsm.py:95: TracerWarning: Converting a tensor to a Python index might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  x1, x2 = x[:, : c // 8], x[:, c // 8:]


* Load from ONNX:

In [6]:
with open(ONNX_MODEL_PATH, 'rb') as onnx_model_file:
    onnx_model = onnx.load_model(onnx_model_file)

onnx.checker.check_model(onnx_model)

In [7]:
print(onnx.helper.printable_graph(onnx_model.graph))

graph torch-jit-export (
  %i0[FLOAT, 1x3x224x224]
  %i1[FLOAT, 1x3x56x56]
  %i2[FLOAT, 1x4x28x28]
  %i3[FLOAT, 1x4x28x28]
  %i4[FLOAT, 1x8x14x14]
  %i5[FLOAT, 1x8x14x14]
  %i6[FLOAT, 1x8x14x14]
  %i7[FLOAT, 1x12x14x14]
  %i8[FLOAT, 1x12x14x14]
  %i9[FLOAT, 1x20x7x7]
  %i10[FLOAT, 1x20x7x7]
) initializers (
  %classifier.bias[FLOAT, 27]
  %classifier.weight[FLOAT, 27x1280]
  %features.0.0.weight[FLOAT, 32x3x3x3]
  %features.0.1.bias[FLOAT, 32]
  %features.0.1.num_batches_tracked[INT64, scalar]
  %features.0.1.running_mean[FLOAT, 32]
  %features.0.1.running_var[FLOAT, 32]
  %features.0.1.weight[FLOAT, 32]
  %features.1.conv.0.weight[FLOAT, 32x1x3x3]
  %features.1.conv.1.bias[FLOAT, 32]
  %features.1.conv.1.num_batches_tracked[FLOAT, scalar]
  %features.1.conv.1.running_mean[FLOAT, 32]
  %features.1.conv.1.running_var[FLOAT, 32]
  %features.1.conv.1.weight[FLOAT, 32]
  %features.1.conv.3.weight[FLOAT, 16x32x1x1]
  %features.1.conv.4.bias[FLOAT, 16]
  %features.1.conv.4.num_batches_tracke

* [optional] Simplify:

In [8]:
model_simp, check = simplify(onnx_model)
assert check, "Simplified ONNX model could not be validated"

In [9]:
print('Before', onnx_model.ByteSize())
print('After', model_simp.ByteSize())

Before 9211341
After 8981232


* ONNX runtime check:

In [10]:
import onnxruntime

ort_session = onnxruntime.InferenceSession(ONNX_MODEL_PATH)
ort_session

* ONNX TensorFlow check:

In [11]:
tf_rep = prepare(onnx_model)

2020-06-27 17:08:55,705 - onnx-tf - INFO - Fail to get since_version of BitShift in domain `` with max_inclusive_version=10. Set to 1.
2020-06-27 17:08:55,707 - onnx-tf - INFO - Unknown op ConstantFill in domain `ai.onnx`.
2020-06-27 17:08:55,707 - onnx-tf - INFO - Fail to get since_version of CumSum in domain `` with max_inclusive_version=10. Set to 1.
2020-06-27 17:08:55,708 - onnx-tf - INFO - Fail to get since_version of Det in domain `` with max_inclusive_version=10. Set to 1.
2020-06-27 17:08:55,709 - onnx-tf - INFO - Fail to get since_version of DynamicQuantizeLinear in domain `` with max_inclusive_version=10. Set to 1.
2020-06-27 17:08:55,710 - onnx-tf - INFO - Fail to get since_version of GatherND in domain `` with max_inclusive_version=10. Set to 1.
2020-06-27 17:08:55,711 - onnx-tf - INFO - Unknown op ImageScaler in domain `ai.onnx`.
2020-06-27 17:08:55,712 - onnx-tf - INFO - Fail to get since_version of Range in domain `` with max_inclusive_version=10. Set to 1.
2020-06-27 1

Instructions for updating:
Create a `tf.sparse.SparseTensor` and use `tf.sparse.to_dense` instead.


In [12]:
print(tf_rep.inputs) # Input nodes to the model
print('-----')
print(tf_rep.outputs) # Output nodes from the model
print('-----')
print(tf_rep.tensor_dict) # All nodes in the model

['i0', 'i1', 'i2', 'i3', 'i4', 'i5', 'i6', 'i7', 'i8', 'i9', 'i10']
-----
['o0', 'o1', 'o2', 'o3', 'o4', 'o5', 'o6', 'o7', 'o8', 'o9', 'o10']
-----
{'classifier.bias': <tf.Tensor 'classifier.bias:0' shape=(27,) dtype=float32>, 'classifier.weight': <tf.Tensor 'classifier.weight:0' shape=(27, 1280) dtype=float32>, 'features.0.0.weight': <tf.Tensor 'features.0.0.weight:0' shape=(32, 3, 3, 3) dtype=float32>, 'features.0.1.bias': <tf.Tensor 'features.0.1.bias:0' shape=(32,) dtype=float32>, 'features.0.1.num_batches_tracked': <tf.Tensor 'features.0.1.num_batches_tracked:0' shape=() dtype=int64>, 'features.0.1.running_mean': <tf.Tensor 'features.0.1.running_mean:0' shape=(32,) dtype=float32>, 'features.0.1.running_var': <tf.Tensor 'features.0.1.running_var:0' shape=(32,) dtype=float32>, 'features.0.1.weight': <tf.Tensor 'features.0.1.weight:0' shape=(32,) dtype=float32>, 'features.1.conv.0.weight': <tf.Tensor 'features.1.conv.0.weight:0' shape=(32, 1, 3, 3) dtype=float32>, 'features.1.conv.1.

In [13]:
TF_MODEL_PATH = './models/jestnet_tf.pb'
tf_rep.export_graph(TF_MODEL_PATH)

Can help: https://pytorch.org/docs/master/onnx.html#tracing-vs-scripting

* Run ONNX model in camera loop:

In [14]:
# print("Open camera...")
# cap = cv2.VideoCapture(0)
# print(cap)
# # set a lower resolution for speed up
# cap.set(cv2.CAP_PROP_FRAME_WIDTH, 320)
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 240)

# t = None
# index = 0
# print("Build buffer...")
# transform = get_transform()
# buffer = (
#     torch.zeros([1, 3, 56, 56]),
#     torch.zeros([1, 4, 28, 28]),
#     torch.zeros([1, 4, 28, 28]),
#     torch.zeros([1, 8, 14, 14]),
#     torch.zeros([1, 8, 14, 14]),
#     torch.zeros([1, 8, 14, 14]),
#     torch.zeros([1, 12, 14, 14]),
#     torch.zeros([1, 12, 14, 14]),
#     torch.zeros([1, 20, 7, 7]),
#     torch.zeros([1, 20, 7, 7])
# )
# idx = 0
# history = [2, 2]
# history_logit = []
# history_timing = []
# i_frame = -1
# print("Camera Ready!")

# while True:
#     i_frame += 1
#     _, img = cap.read()
#     if i_frame % 2 == 0:
#         t1 = time.time()
#         img_tran = transform([Image.fromarray(img).convert('RGB')])
#         input_var = torch.autograd.Variable(img_tran.view(1, 3, img_tran.size(1), img_tran.size(2)))
#         with torch.no_grad():
#             outputs = torch_module(input_var, *buffer)
#             feat, buffer = outputs[0], outputs[1:]
#         if SOFTMAX_THRES > 0:
#             feat_np = feat.numpy().reshape(-1)
#             feat_np -= feat_np.max()
#             softmax = np.exp(feat_np) / np.sum(np.exp(feat_np))
#             print(max(softmax))
#             if max(softmax) > SOFTMAX_THRES:
#                 idx_ = np.argmax(feat.numpy(), axis=1)[0]
#             else:
#                 idx_ = idx
#         else:
#             idx_ = np.argmax(feat.numpy(), axis=1)[0]
#         if HISTORY_LOGIT:
#             history_logit.append(feat.numpy())
#             history_logit = history_logit[-12:]
#             avg_logit = sum(history_logit)
#             idx_ = np.argmax(avg_logit, axis=1)[0]
#         idx, history = process_output(idx_, history)
#         t2 = time.time()
# #             print(f"{index} {catigories[idx]}")
#         current_time = t2 - t1

#     print(f"CV EVENT: {index} {catigories[idx]}")

#     if t is None:
#         t = time.time()
#     else:
#         nt = time.time()
#         index += 1
#         t = nt